### Import Deps

In [ ]:
import random
from qdrant_client import QdrantClient

import instructor
import json
from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken #how we can calculate number of tokens in a specific piece of text

In [ ]:
### Load Environment
from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Load all data from qdrant

In [ ]:
all_points = []

next_offset = None
batch_size = 100

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        limit = batch_size,
        offset = next_offset,
        with_vectors = False,
        with_payload = True
    )

    all_points.extend(points)
    if next_offset is None:
        break
    print(f"Retrieved {len(points)} points, next_offset: {next_offset}")


### Split the data into 2 groups: 
### 50 items for synthetic question generation
### 950 to loop and evaluate their relevance for the previously generated synthetic questions


In [ ]:
rng = random.Random(42) # RANDOM SEED USING 42
indeces = list(range(len(all_points)))
rng.shuffle(indeces)

# synthetic_question_items = [all_points[i] for i in indeces[:50]]
# evaluation_items = [all_points[i] for i in indeces[50:]]

sample_idx = set(indeces[:50])
sample_50 = [p for i, p in enumerate(all_points) if i in sample_idx]
remaining_points = [p for i, p in enumerate(all_points) if i not in sample_idx]

len(sample_50), len(remaining_points)

### Generate synthetic questions

In [ ]:
all_context_sample50 = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]

In [ ]:
len(all_context_sample50)

In [ ]:
all_context_remaining = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in remaining_points]

In [ ]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant pro

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** - answerable using exactly ONE product.
2. **Multi-product (10 questions)** - answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** - plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that have a large number of products.

## Products
{json.dumps(all_context_sample50, indent=2)}
"""

# we will create an expecte shapre for response with instructor probably


In [ ]:
len(all_context_sample50)

In [ ]:
class EvalQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks")
    question: str = Field(description="Suggested question.")
    chunk_ids:list[str] = Field(description="IDs of the chunks that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
    synthetic_questions: list[EvalQuestion] = Field(description="List of synthetic questions generated for the evaluation.")


In [ ]:
client = instructor.from_provider("openai/gpt-5.4", mode=instructor.Mode.RESPONSES_TOOLS)

response, raw_response = client.create_with_completion(
    messages=[
        {"role": "user", "content": SYSTEM_PROMPT}
    ],
    reasoning={"effort": "low"},
    response_model=SyntheticQuestions,
)


In [ ]:
for s_question in response.synthetic_questions:
    print(f"Question: {s_question.chunk_ids}")

In [ ]:
synthetic_questions_relevant_data = [
    {
        "question_id": i,
        "question": item.question
    }
    for i, item in enumerate(response.synthetic_questions)
]

### assign remaining chunks to relevant questions

In [ ]:
SYSTEM_PROMPT_950 = f"""
You're a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_context_remaining[0:50], indent=2)}
"""

In [ ]:
print(len(all_context_remaining[0:50]))

### Count the number of tokens for the static prefix

In [ ]:
SYSTEM_PROMPT_STATIC_PART = SYSTEM_PROMPT_950 = f"""
You're a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}
"""

In [ ]:
len(synthetic_questions_relevant_data)

In [ ]:
### we will use tiktoken

def count_tokens(text):
    encoding = tiktoken.get_encoding("o200k_base")
    return len(encoding.encode(text))

In [ ]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART)) # -> 1345 for gpt fam models = safe because we need 1024 for caching ot kick in 

### Assign chunks to relevant questions for a single batch

In [ ]:
print(len(all_context_remaining),
len(synthetic_questions_relevant_data))

In [ ]:
mini_client = instructor.from_provider("openai/gpt-5.4-mini", mode=instructor.Mode.RESPONSES_TOOLS)

In [ ]:
class SyntheticQuestionMatchingChunks(BaseModel):
    reasoning: str = Field(description="Reasoning behind the selection of chunk IDs.")
    question_id: int = Field(description="Unique identifier for the evaluation question.")
    chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class SyntheticQuestionMatchingChunksList(BaseModel):
    synthetic_questions: list[SyntheticQuestionMatchingChunks] = Field(description="List of synthetic questions")


In [ ]:
tmp_response, raw_response = mini_client.create_with_completion(
    messages=[
        {"role": "user", "content": SYSTEM_PROMPT_950}
    ],
    reasoning={"effort": "low"},
    response_model=SyntheticQuestions,
)


In [ ]:
len(response.synthetic_questions)

In [ ]:
for i, q in enumerate(response.synthetic_questions):
    print(f"Q{i}: {q.chunk_ids}")
# so now i assume will append to the chunks

In [ ]:

def get_relevant_chunks(synthetic_questions, remaining_points_batch):
    SYSTEM_PROMPT_950_LOCAL = f"""
    You're a relevance annotator building ground-truth labels for a RAG retrieval system.

    ## System being tested
    The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

    ## Your input
    1. A list of customer questions, each with an `id`.
    2. A list of 50 products, each with an `id` and a `text` description.

    ## Your task
    For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

    - Include a product ID only if that product ALONE is sufficient to answer the question on its own.
    - Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
    - If no product independently answers the question, return an empty list.
    - I need synthetic_questions to return an entry for every question, even if its chunk_ids list is empty.

    ## Questions
    {json.dumps(synthetic_questions, indent=2)}

    ## Products
    {json.dumps(remaining_points_batch, indent=2)}
    """
    print(len(synthetic_questions))
    ext_chunks_res, raw_response = mini_client.create_with_completion(
        messages=[
            {"role": "user", "content": SYSTEM_PROMPT_950_LOCAL}
        ],
        reasoning={"effort": "low"},
        response_model=SyntheticQuestionMatchingChunksList,
    )
    return ext_chunks_res.synthetic_questions


In [ ]:
answer_extended_chunks = []
for i in range(0, len(all_context_remaining), 50):
    print(f"Working on batch {i//50 + 1}")
    remaining_points_batch = all_context_remaining[i:i+50]
    print(f"length of remaining points batch {len(remaining_points_batch)}, synthetic question count: {len(synthetic_questions_relevant_data)}")
    extended_chunks = get_relevant_chunks(synthetic_questions_relevant_data, remaining_points_batch)
    print(f"length of extended chunks {len(extended_chunks)}")
    print(extended_chunks)
    answer_extended_chunks.append(extended_chunks)

In [ ]:
len(answer_extended_chunks[0])

In [ ]:
# answer_extended_chunks

In [ ]:
# # Build a merged copy of synthetic questions with all chunk_ids collected across batches
# merged_questions = [
#     {
#         "question_id": i,
#         "question": q.question,
#         "chunk_ids": list(q.chunk_ids),
#         "answer_example": q.answer_example,
#         "reasoning": q.reasoning,
#     }
#     for i, q in enumerate(response.synthetic_questions)
# ]

# question_map = {q["question_id"]. : q for q in merged_questions}

# for batch in answer_extended_chunks:
#     for item in batch:
#         if item.question_id in question_map:
#             existing = set(question_map[item.question_id]["chunk_ids"])
#             for chunk_id in item.chunk_ids:
#                 if chunk_id not in existing:
#                     question_map[item.question_id]["chunk_ids"].append(chunk_id)
#                     existing.add(chunk_id)

# # preview chunk counts per question
# for q in merged_questions:
#     print(f"Q{q['question_id']}: {len(q['chunk_ids'])} chunks -> {q['chunk_ids']}")


In [ ]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

In [ ]:
questions_container = {i: [] for i in range(len(synthetic_questions_relevant_data))} # {0: [], 1: []}

In [ ]:
questions_container

In [ ]:
for i, data in enumerate(answer_extended_chunks):
    for item in data:
        questions_container[item.question_id].extend(item.chunk_ids)

In [ ]:
questions_container

class EvalQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks")
    question: str = Field(description="Suggested question.")
    chunk_ids:list[str] = Field(description="IDs of the chunks that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
    synthetic_questions: list[EvalQuestion] = Field(description="List of synthetic questions generated for the evaluation.")

In [ ]:
for i, data in enumerate(response.synthetic_questions):
    questions_container[i].extend(data.chunk_ids)

In [ ]:
questions_container

In [ ]:
def get_point_by_parent_asin(parent_asin):
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]
    return points[0].payload["preprocessed_description"]

In [ ]:
print(get_point_by_parent_asin("B0B1DBFRWL"))

In [ ]:
def generate_golden_answer(question, relevant_chunks):
    class GoldenAnswer(BaseModel):
        answer: str = Field(description = 'The ideal reference answer for this question')
        
    
    
    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as a ground truth for evluating a RAG system

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer 
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the information isn't available (e.g politely say you couldn't find it), and do not fabricate an answer

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "user", "content": PROMPT}
        ],
        reasoning={"effort": "low"},
        response_model=GoldenAnswer,
    )
    return response.answer

    
    

In [ ]:
reference_dataset = []

for i, data in enumerate(response.synthetic_questions):
    print(f"Working on question {i+1}: {data.question}")
    question = data.question
    relevant_chunks = [
        get_point_by_parent_asin(item) for item in questions_container[i]
    ]
    print(relevant_chunks)
    golden_answer = generate_golden_answer(data.question, relevant_chunks)
    reference_dataset.append({
        'question': question,
        'golden_answer': golden_answer,
        'reference_context_ids': questions_container[i],
        'reference_descriptions': relevant_chunks,
    })
    
    
    

In [ ]:
reference_dataset[0]

### Create langsmith dataset

In [ ]:
ls_client = Client()

dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset for 1000 items"
)
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["golden_answer"],
            "reference_context_ids": item["reference_context_ids"],
            "reference_descriptions": item["reference_descriptions"]
        }
    )